# 🚗 YOLO11n Hassas İnce Ayar (Fine-Tuning) ve Araç Tespiti Eğitimi
### Bitirme Projesi - Yıkıcı Unutmayı (Catastrophic Forgetting) Önleyen İyileştirilmiş Eğitim

Bu notebook, **`vehicle_dataset_v3_augmented`** veri kümesini kullanarak **YOLO11 Nano (`yolo11n`)** modelini omurga katmanlarını dondurarak (freeze=10) ve öğrenme oranını hassaslaştırarak eğitmek için hazırlanmıştır.

**Bu Sürümdeki Kritik İyileştirmeler:**
1. **`freeze=10` (Omurga Dondurma):** Modelin COCO 118K veri setinden öğrendiği genel araç/tekerlek/kenar algılama yeteneği silinmez (Yıkıcı Unutma engellenir).
2. **`lr0=0.001` (Düşük Öğrenme Oranı):** Ağırlıkların bozulmasını önlemek için öğrenme hızı 10 kat düşürülmüştür.
3. **`cos_lr=True` & `warmup_epochs=3`:** Dengeli ve yumuşak öğrenme oranı geçişleri.
4. **Otomatik Paketleme:** Eğitim bittiğinde sonuçlar ve `best.pt` otomatik olarak `yolo11_results.zip` adıyla bilgisayarınıza indirilir.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aydin-eng/federated-traffic-learning/blob/main/traffic_vehicle_detection_yolo11.ipynb)


## 1. Donanım ve GPU Kontrolü
Google Colab'de GPU (T4 / V100 / A100) hızlandırıcısının devrede olduğunu doğruluyoruz.


In [ ]:
!nvidia-smi


## 2. Deponun Klonlanması ve Çalışma Ortamı Kurulumu
GitHub üzerindeki güncel veri setimizi ve yapılandırma dosyalarımızı Colab ortamına çekiyoruz.


In [ ]:
import os

# Eğer Colab üzerinde çalışıyorsak repoyu klonla
if not os.path.exists('federated-traffic-learning'):
    !git clone https://github.com/aydin-eng/federated-traffic-learning.git
    %cd federated-traffic-learning
else:
    print('Repo zaten mevcut, güncelleniyor...')
    %cd federated-traffic-learning
    !git pull

print('Mevcut çalışma dizini:', os.getcwd())


## 3. Gerekli Kütüphanelerin Yüklenmesi
Ultralytics (YOLO11) kütüphanesini yüklüyoruz.


In [ ]:
!pip install -q ultralytics


## 4. Veri Seti ve YAML Konfigürasyonu Doğrulama
`vehicle_dataset_v3_augmented` veri setimizin ve `data.yaml` dosyamızın yollarını kontrol ediyoruz.


In [ ]:
import yaml

yaml_path = 'vehicle_dataset_v3_augmented/data.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print('Veri Konfigürasyonu:')
for k, v in cfg.items():
    print(f' - {k}: {v}')


## 5. YOLO11n Modelinin Başlatılması
Ultralytics üzerinden önceden eğitilmiş temel **YOLO11n (Nano)** model ağırlığını yüklüyoruz.


In [ ]:
from ultralytics import YOLO

# Önceden eğitilmiş YOLO11 Nano modelini yükle
model = YOLO('yolo11n.pt')
print('Base YOLO11n modeli başarıyla yüklendi!')
model.info()


## 6. İyileştirilmiş Model Eğitimi (Fine-Tuning) 🚀
Modelin geçmiş genel araç bilgisini korumak ve aşırı uyumu (overfitting) önlemek için optimize edilmiş hiperparametrelerle eğitimi başlatıyoruz:

* **`freeze=10`**: İlk 10 omurga katmanı dondurulur; genel araç öznitelikleri bozulmaz.
* **`lr0=0.001`**: Öğrenme hızı ince ayar için 10 kat düşürülmüştür.
* **`cos_lr=True`**: Cosine öğrenme oranı zamanlayıcısı.
* **`warmup_epochs=3`**: Başlangıçta ani gradyan sıçramalarını önler.
* **`mosaic=0.5` & `close_mosaic=10`**: Dengeli veri zenginleştirme.
* **`epochs=50`, `imgsz=640`, `batch=16`**


In [ ]:
# İyileştirilmiş eğitimi başlat
results = model.train(
    data='vehicle_dataset_v3_augmented/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,                   # Colab GPU
    project='runs/train',
    name='yolo11n_vehicle_exp',
    freeze=10,                  # Omurga dondurma (Yıkıcı unutmayı önler)
    lr0=0.001,                  # İnce ayar için optimize edilmiş öğrenme hızı
    lrf=0.01,
    cos_lr=True,                # Yumuşak öğrenme eğrisi
    warmup_epochs=3,            # Isınma turları
    mosaic=0.5,                 # Dengeli mozaik artırma
    close_mosaic=10,            # Son 10 epoch mozaik kapatma
    patience=20,                # Erken durdurma toleransı
    save=True,
    plots=True,
    seed=42,
    verbose=True
)


## 7. Model Başarısının Test Kümesinde Değerlendirilmesi (Evaluation)
Eğitilen en iyi ağırlık (`best.pt`) ile test kümesi (`images/test`) üzerindeki mAP50, mAP50-95, Precision ve Recall değerlerini ölçüyoruz.


In [ ]:
# Test kümesi üzerinde değerlendirme
metrics = model.val(split='test')

print('\n' + '=' * 45)
print('  TEST KÜMESİ DEĞERLENDİRME SONUÇLARI')
print('=' * 45)
print(f"mAP@50:     {metrics.box.map50:.4f}")
print(f"mAP@50-95:  {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")
print('=' * 45)


## 8. Eğitim Metrikleri ve Karışıklık Matrisi Grafikleri
Kayıp eğrilerini, PR eğrisini ve sınıflara göre Confusion Matrix grafiğini görüntülüyoruz.


In [ ]:
import glob
from IPython.display import Image, display

plots = [
    'runs/train/yolo11n_vehicle_exp/results.png',
    'runs/train/yolo11n_vehicle_exp/confusion_matrix_normalized.png',
    'runs/train/yolo11n_vehicle_exp/BoxPR_curve.png'
]

for p in plots:
    if os.path.exists(p):
        print(f'Görsel: {os.path.basename(p)}')
        display(Image(filename=p, width=700))


## 9. Test ve Özel Görseller Üzerinde Model Çıkarımı (Inference)
Modelin hem test kümesindeki hem de yüklediğimiz özel test fotoğrafındaki (`test_samples/user_test_photo.jpg`) tespitlerini çizdiriyoruz.


In [ ]:
# 1. Test kümesindeki görseller üzerinde tahmin yap
test_preds = model.predict(
    source='vehicle_dataset_v3_augmented/images/test',
    conf=0.25,
    save=True,
    project='runs/detect',
    name='test_results'
)

# 2. Özel test fotoğrafı varsa onu da tahmin et
custom_photo = 'test_samples/user_test_photo.jpg'
if os.path.exists(custom_photo):
    custom_pred = model.predict(
        source=custom_photo,
        conf=0.25,
        save=True,
        project='runs/detect',
        name='custom_test_result'
    )
    pred_custom_imgs = glob.glob('runs/detect/custom_test_result/*.jpg')
    if pred_custom_imgs:
        print('Özel test fotoğrafı sonucu:')
        display(Image(filename=pred_custom_imgs[0], width=700))

# İlk 2 test kümesi sonucunu görselleştir
pred_images = glob.glob('runs/detect/test_results/*.jpg')
print(f'Toplam {len(pred_images)} test görseli işlendi. Örnek sonuçlar:')
for img_path in pred_images[:2]:
    display(Image(filename=img_path, width=650))


## 10. Sonuçların Otomatik Paketlenmesi ve İndirilmesi 📦
Eğitilen `best.pt` ağırlık dosyasını ve tüm grafikleri tek bir `yolo11_results.zip` dosyası haline getirip bilgisayarınıza indirir.


In [ ]:
import shutil
from google.colab import files

best_weights = 'runs/train/yolo11n_vehicle_exp/weights/best.pt'
if os.path.exists(best_weights):
    size_mb = os.path.getsize(best_weights) / (1024 * 1024)
    print(f'✅ En başarılı model ağırlığı hazır: {best_weights} ({size_mb:.2f} MB)')

print('📦 Sonuçlar paketleniyor ve yolo11_results.zip indiriliyor...')
shutil.make_archive('yolo11_results', 'zip', 'runs/train/yolo11n_vehicle_exp')
files.download('yolo11_results.zip')
print('🚀 yolo11_results.zip indirme başlatıldı! İndirilen dosyayı masaüstündeki proje klasörünüze atabilirsiniz.')
